# Imagenet 64 (64x64x3) -> Julia Memory Mapped IO

In [1]:
include("lib/imagenet.jl")
include("lib/utils.jl")

import .ImageNet
using .Utils

function load_one(relpath::String)
    # N 64 64 3
    labels, images, μ = ImageNet.load_64(relpath)
    # 64 64 3 N
    pimages = permutedims(images, (2, 3, 4, 1))
    return (labels, pimages, μ)
end

function load(files, output)
    base = "build/$output"
    mkpath(base)
	open("$base/count.ser", "w") do io
		write(io, length(files))
	end

	ys = open("$base/ys.ser", "w")
	Xs = open("$base/Xs.ser", "w")
	μs = open("$base/μs.ser", "w")
	for (i, f) ∈ enumerate(files)
		ser = load_one(f)
		y, X, μ = ser
		serialize(Xs, X)
		serialize(μs, μ)
		serialize(ys, y)

		h, w, c, n = size(X)
		@info "input" file=f n=n
	end
	close(ys)
	close(Xs)
	close(μs)
end

load (generic function with 1 method)

# Training Dataset

In [2]:
@time begin
    part1 = readdir("$(ImageNet.IMAGENET_64)/Imagenet64_train_part1")
    part2 = readdir("$(ImageNet.IMAGENET_64)/Imagenet64_train_part2")
    
    files = vcat(
        ["Imagenet64_train_part1/$f" for f ∈ part1],
        ["Imagenet64_train_part2/$f" for f ∈ part2],
    )
    load(files, "train")
end

┌ Info: input
│   file = "Imagenet64_train_part1/train_data_batch_1"
└   n = 128116
┌ Info: input
│   file = "Imagenet64_train_part1/train_data_batch_2"
└   n = 128116
┌ Info: input
│   file = "Imagenet64_train_part1/train_data_batch_3"
└   n = 128116
┌ Info: input
│   file = "Imagenet64_train_part1/train_data_batch_4"
└   n = 128116
┌ Info: input
│   file = "Imagenet64_train_part1/train_data_batch_5"
└   n = 128116
┌ Info: input
│   file = "Imagenet64_train_part2/train_data_batch_10"
└   n = 128123
┌ Info: input
│   file = "Imagenet64_train_part2/train_data_batch_6"
└   n = 128116
┌ Info: input
│   file = "Imagenet64_train_part2/train_data_batch_7"
└   n = 128116
┌ Info: input
│   file = "Imagenet64_train_part2/train_data_batch_8"
└   n = 128116
┌ Info: input
│   file = "Imagenet64_train_part2/train_data_batch_9"
└   n = 128116


188.962846 seconds (70.96 M allocations: 207.561 GiB, 4.68% gc time, 1.19% compilation time)


# Validation Dataset

In [ ]:
@time begin
    files = readdir("$(ImageNet.IMAGENET_64)/Imagenet64_val")
    load(files, "val")
end